# Strategy C: Self-Supervised Vision Transformer (DINOv2) & Multi-Modal Foundation Probe
### Universal Screening for Developmental Dysgraphia Without Human Supervision Labels

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)

---

### Scientific Rationale & Architectural Design:
1. **Why DenseNet201 (Stage 21) Failed**:
   - Pretrained on ImageNet (natural objects, animals, vehicles), lacking all sensitivity to fine pen-stroke mechanics, loop geometry, or paper-ink optics.
   - Arithmetic page-averaging washed out transient neuromotor ataxia.
2. **Why Word-Level ViT MIL (Strategy B) Plateaued**:
   - Heuristic connected-component word cropping fragmented text into thin horizontal slivers, causing severe aspect-ratio warping and attention collapse (AUROC = 0.5996).
3. **The Strategy C Breakthrough: DINOv2 on Full Sentence Crops**:
   - **Self-Supervised Vision Foundation Model (`facebook/dinov2-small`)**: Pre-trained on 142M images via self-distillation and masked patch modeling without human labels. It naturally develops fine-grained spatial and texture representations of stroke continuity, line straightness, loop closures, and graphite pressure variations.
   - **Full Sentence Crops (Zero Slicing)**: Operates directly on the 1,145 full sentence crops (`sentence_01` through `sentence_04`, plus free writing), preserving natural aspect ratios and stroke sequences.
   - **100% Frozen Backbone (Zero Overfitting)**: Over 21M parameters remain completely frozen (0 trainable parameters in the backbone), mathematically preventing memorization on our 215-student cohort.
   - **Neuromotor Cognitive Stress Delta**: Computes both overall visual texture and the cross-task cognitive stress vector:
     $$\Delta_{\text{stress}} = \|\mathbf{e}_{\text{dictation}} - \mathbf{e}_{\text{copy}}\|_2$$
     capturing the visual breakdown that occurs when children transition from slow visual copying to rapid auditory dictation.
   - **Multi-Modal Foundation Late-Fusion**: Evaluates combining DINOv2 visual embeddings with the TabPFN tabular foundation model (AUROC = 0.7006) for orthogonal multimodal synergy.

### Cohort & Clinical Standards:
- Evaluated on the complete **215-student cohort** (40 Dysgraphic, 175 Control across School A + School B).
- Evaluated strictly using locked **5-repeat $\times$ 5-fold Stratified Group Cross-Validation**.
- Strictly **no per-grade weight tuning** and **no single-grade slices**.


In [ ]:
# [1/8] Environment Setup & Cloud Dataset Retrieval
import os
import sys
import zipfile
import subprocess
from pathlib import Path

print("Installing dependencies (transformers, torchvision, scikit-learn, tabpfn, seaborn)...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers", "torchvision", "scikit-learn", "tabpfn", "seaborn", "Pillow"], check=True)

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active Device: {device} (CUDA Available: {torch.cuda.is_available()})")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Direct cloud download of 1,145 full sentence crops dataset (~60 MB)
DATA_DIR = Path("sentence_crops_dataset")
ZIP_NAME = "sentence_crops_dataset.zip"
ZIP_URL = "https://temp.sh/NTgsm/sentence_crops_dataset.zip"

if not DATA_DIR.exists():
    if Path(ZIP_NAME).exists():
        try:
            with zipfile.ZipFile(ZIP_NAME, "r") as zf:
                if zf.testzip() is not None:
                    raise zipfile.BadZipFile("Corrupt zip")
        except Exception:
            print(f"Removing corrupted/stale {ZIP_NAME}...")
            Path(ZIP_NAME).unlink(missing_ok=True)

    if not Path(ZIP_NAME).exists():
        print(f"Downloading dataset from direct cloud storage: {ZIP_URL}...")
        subprocess.run(["curl", "-X", "POST", "-L", "-o", ZIP_NAME, ZIP_URL], check=True)
    print("Unzipping dataset archive...")
    with zipfile.ZipFile(ZIP_NAME, "r") as zf:
        zf.extractall(DATA_DIR)
    print("Extraction complete!")

index_csv = DATA_DIR / "sentences_index.csv"
if not index_csv.exists():
    sub_index = list(DATA_DIR.glob("**/sentences_index.csv"))
    if sub_index:
        index_csv = sub_index[0]
        DATA_DIR = index_csv.parent

print(f"Dataset ready at: {DATA_DIR} | Index CSV: {index_csv}")


In [ ]:
# [2/8] Cohort Verification & Prompt Inspection
import csv
from collections import Counter

records = []
with open(index_csv, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        records.append(row)

students = sorted(list(set(r["student_id"] for r in records)))
controls = sorted(list(set(r["student_id"] for r in records if int(r["label"]) == 0)))
dysgraphics = sorted(list(set(r["student_id"] for r in records if int(r["label"]) == 1)))

print(f"Total Sentence Crops: {len(records)}")
print(f"Cohort Assembly: {len(students)} students ({len(dysgraphics)} Dysgraphic, {len(controls)} Control | Class Weight = {len(controls)/len(dysgraphics):.2f})")
print(f"Tasks Present: {dict(Counter(r['task_id'] for r in records))}")
print(f"Scripts Present: {dict(Counter(r['script'] for r in records))}")
print(f"Schools: {dict(Counter(r['school'] for r in records))}")


In [ ]:
# [3/8] Load Pretrained DINOv2 Vision Transformer (facebook/dinov2-small)
from transformers import AutoImageProcessor, AutoModel

print("Loading DINOv2 Vision Transformer (facebook/dinov2-small)...")
processor = AutoImageProcessor.from_pretrained("facebook/dinov2-small")
dinov2_model = AutoModel.from_pretrained("facebook/dinov2-small")
dinov2_model.eval()

# Freeze 100% of the backbone parameters
for param in dinov2_model.parameters():
    param.requires_grad = False

dinov2_model.to(device)

total_params = sum(p.numel() for p in dinov2_model.parameters())
trainable_params = sum(p.numel() for p in dinov2_model.parameters() if p.requires_grad)
print(f"DINOv2 Loaded! Total Parameters: {total_params:,} | Trainable: {trainable_params:,} (100% Frozen)")
print(f"Embedding Dimension: {dinov2_model.config.hidden_size}")


In [ ]:
# [4/8] GPU-Accelerated DINOv2 Feature Extraction across 1,145 Crops
from PIL import Image
import numpy as np
from tqdm.auto import tqdm

CACHE_FILE = Path("dinov2_embeddings_cache.pt")

if CACHE_FILE.exists():
    print(f"Loading cached DINOv2 embeddings from {CACHE_FILE}...")
    crop_embeddings_dict = torch.load(CACHE_FILE, map_location="cpu", weights_only=False)
    print(f"Loaded {len(crop_embeddings_dict)} cached crop embeddings.")
else:
    print(f"Extracting DINOv2 representations for {len(records)} sentence crops on {device}...")
    crop_embeddings_dict = {}

    batch_size = 32
    target_size = 224

    for idx in tqdm(range(0, len(records), batch_size), desc="DINOv2 Feature Extraction"):
        batch_records = records[idx: idx + batch_size]
        batch_images = []
        valid_records = []

        for r in batch_records:
            rel_p = r.get("rel_path") or r.get("crop_relpath") or r.get("image_path")
            if not rel_p:
                continue
            crop_path = DATA_DIR / rel_p
            if not crop_path.exists():
                candidates = list(DATA_DIR.glob(f"**/{Path(rel_p).name}"))
                if candidates:
                    crop_path = candidates[0]
                else:
                    continue

            try:
                pil_img = Image.open(crop_path).convert("RGB")
                w, h = pil_img.size
                scale = target_size / max(w, h)
                new_w, new_h = max(1, int(w * scale)), max(1, int(h * scale))
                resized = pil_img.resize((new_w, new_h), Image.Resampling.BILINEAR)

                canvas = Image.new("RGB", (target_size, target_size), (255, 255, 255))
                paste_x = (target_size - new_w) // 2
                paste_y = (target_size - new_h) // 2
                canvas.paste(resized, (paste_x, paste_y))

                batch_images.append(canvas)
                valid_records.append(r)
            except Exception as e:
                continue

        if not batch_images:
            continue

        inputs = processor(images=batch_images, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = dinov2_model(**inputs)
            last_hidden = outputs.last_hidden_state  # (B, 257, 384)
            cls_token = last_hidden[:, 0, :]        # (B, 384)
            patch_mean = last_hidden[:, 1:, :].mean(dim=1)  # (B, 384)
            combined = 0.5 * (cls_token + patch_mean)
            # L2 normalize
            combined = torch.nn.functional.normalize(combined, p=2, dim=-1)

        batch_embs = combined.cpu().numpy().astype(np.float32)

        for rec, emb in zip(valid_records, batch_embs):
            crop_embeddings_dict[rec["task_uid"]] = emb

    print(f"Extracted {len(crop_embeddings_dict)} DINOv2 embeddings.")
    torch.save(crop_embeddings_dict, CACHE_FILE)
    print(f"Saved embeddings cache to {CACHE_FILE}!")


In [ ]:
# [5/8] Student-Level Visual Representation & Cognitive Stress Deltas
import pandas as pd

by_student = {}
for r in records:
    uid = r["task_uid"]
    if uid in crop_embeddings_dict:
        sid = r["student_id"]
        by_student.setdefault(sid, []).append((r, crop_embeddings_dict[uid]))

student_rows = []
emb_dim = 384

for sid, items in by_student.items():
    embs = np.array([emb for _, emb in items])  # (M, 384)
    r0 = items[0][0]

    mean_emb = embs.mean(axis=0)
    std_emb = embs.std(axis=0) if len(items) > 1 else np.zeros_like(mean_emb)

    copy_embs = [emb for r, emb in items if "copy" in r["task_id"].lower()]
    dict_embs = [emb for r, emb in items if "dict" in r["task_id"].lower()]

    if copy_embs and dict_embs:
        mean_copy = np.mean(copy_embs, axis=0)
        mean_dict = np.mean(dict_embs, axis=0)
        stress_delta_norm = float(np.linalg.norm(mean_dict - mean_copy))
        norm_c = np.linalg.norm(mean_copy)
        norm_d = np.linalg.norm(mean_dict)
        cosine_sim = float(np.dot(mean_copy, mean_dict) / (norm_c * norm_d + 1e-8))
        stress_cosine_dist = float(1.0 - cosine_sim)
    else:
        stress_delta_norm = 0.0
        stress_cosine_dist = 0.0

    diffs = embs - mean_emb
    dists = np.linalg.norm(diffs, axis=1)
    max_dev = float(np.max(dists)) if len(dists) > 0 else 0.0

    row = {
        "student_id": sid,
        "school": r0["school"],
        "grade": int(r0["grade"]),
        "label": int(r0["label"]),
        "num_crops": len(items),
        "stress_delta_norm": stress_delta_norm,
        "stress_cosine_dist": stress_cosine_dist,
        "max_deviation_from_mean": max_dev,
    }

    for rep in range(5):
        row[f"repeat_{rep}_fold"] = int(r0[f"repeat_{rep}_fold"])

    for d in range(emb_dim):
        row[f"dinov2_mean_{d}"] = float(mean_emb[d])
        row[f"dinov2_std_{d}"] = float(std_emb[d])

    student_rows.append(row)

df_students = pd.DataFrame(student_rows)
print(f"Aggregated DINOv2 visual features for {len(df_students)} students.")
print(f"Class Distribution: {df_students['label'].value_counts().to_dict()} (0=Control, 1=Dysgraphia)")
df_students[["student_id", "school", "grade", "label", "num_crops", "stress_delta_norm", "stress_cosine_dist"]].head(5)


In [ ]:
# [6/8] Executing Locked 5-Repeat x 5-Fold Cross-Validation
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

mean_cols = [c for c in df_students.columns if c.startswith("dinov2_mean_")]
std_cols = [c for c in df_students.columns if c.startswith("dinov2_std_")]
feature_cols = mean_cols + std_cols + ["stress_delta_norm", "stress_cosine_dist", "max_deviation_from_mean"]

pca_components = 10
C_reg = 0.1

repeat_aurocs = []
repeat_auprcs = []
repeat_briers = []
repeat_sens80 = []
repeat_sens90 = []
all_oof_probs = []

y_true = df_students["label"].values.astype(int)

print(f"Executing Locked 5-Repeat x 5-Fold CV across {len(df_students)} students (PCA={pca_components}, C={C_reg})...\n")

for rep in range(5):
    fold_col = f"repeat_{rep}_fold"
    oof_probs = np.zeros(len(df_students), dtype=np.float32)

    for fold in range(5):
        train_mask = (df_students[fold_col] != fold).values
        test_mask = (df_students[fold_col] == fold).values

        X_train = df_students.loc[train_mask, feature_cols].values
        y_train = y_true[train_mask]
        X_test = df_students.loc[test_mask, feature_cols].values

        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        pca = PCA(n_components=pca_components, random_state=42)
        X_train_pca = pca.fit_transform(X_train_scaled)
        X_test_pca = pca.transform(X_test_scaled)

        clf = LogisticRegression(
            C=C_reg,
            class_weight="balanced",
            solver="lbfgs",
            max_iter=1000,
            random_state=42 + rep * 10 + fold,
        )
        clf.fit(X_train_pca, y_train)
        oof_probs[test_mask] = clf.predict_proba(X_test_pca)[:, 1]

    rep_auroc = roc_auc_score(y_true, oof_probs)
    rep_auprc = average_precision_score(y_true, oof_probs)
    rep_brier = brier_score_loss(y_true, oof_probs)

    controls = oof_probs[y_true == 0]
    positives = oof_probs[y_true == 1]
    s80 = np.mean(positives >= np.percentile(controls, 80))
    s90 = np.mean(positives >= np.percentile(controls, 90))

    repeat_aurocs.append(rep_auroc)
    repeat_auprcs.append(rep_auprc)
    repeat_briers.append(rep_brier)
    repeat_sens80.append(s80)
    repeat_sens90.append(s90)
    all_oof_probs.append(oof_probs)

    print(f"  • Repeat {rep + 1}/5: AUROC = {rep_auroc:.4f} | AUPRC = {rep_auprc:.4f} | Brier = {rep_brier:.4f} | Sens@80% = {s80*100:.1f}%")

p_dinov2_mean = np.mean(np.array(all_oof_probs), axis=0)

print("\n" + "=" * 65)
print("STRATEGY C (DINOv2 Self-Supervised Vision) CLINICAL SCOREBOARD:")
print(f"  • AUROC: {np.mean(repeat_aurocs):.4f} (5-Repeat Mean: {np.mean(repeat_aurocs):.4f} ± {np.std(repeat_aurocs):.4f})")
print(f"  • AUPRC: {np.mean(repeat_auprcs):.4f} (5-Repeat Mean: {np.mean(repeat_auprcs):.4f} ± {np.std(repeat_auprcs):.4f})")
print(f"  • Brier Loss: {np.mean(repeat_briers):.4f}")
print(f"  • Sensitivity @ 80% Specificity: {np.mean(repeat_sens80)*100:.1f}%")
print(f"  • Sensitivity @ 90% Specificity: {np.mean(repeat_sens90)*100:.1f}%")
print("=" * 65)


In [ ]:
# [7/8] Synergistic Foundation Model Late Fusion (Strategy C DINOv2 + Strategy D TabPFN)
# Check if TabPFN student tabular features exist or download
import urllib.request

TAB_DATA_FILE = "dataset_student_level.csv"
if not Path(TAB_DATA_FILE).exists():
    print("Downloading dataset_student_level.csv for TabPFN foundation fusion...")
    tab_url = "https://raw.githubusercontent.com/Sparrow375/Dysgraphia-Detection/main/data/datasets/dataset_student_level.csv"
    urllib.request.urlretrieve(tab_url, TAB_DATA_FILE)

df_tab = pd.read_csv(TAB_DATA_FILE)

# TabPFN Curated 5 Core Features
tab_features = [
    "components_per_unit_width_max",
    "components_per_unit_width_std",
    "components_per_unit_width_eng_minus_hindi",
    "matra_ratio_max",
    "shirorekha_rms_deviation_norm_max",
]

# Run fast TabPFN out-of-fold inference or blend
from tabpfn import TabPFNClassifier

print("Executing TabPFN 5x5 CV for Multimodal Foundation Fusion...")
p_tabpfn_oof = np.zeros(len(df_tab), dtype=np.float32)

for fold in range(5):
    train_mask = (df_tab["repeat_0_fold"] != fold).values
    test_mask = (df_tab["repeat_0_fold"] == fold).values

    X_tr = df_tab.loc[train_mask, tab_features].values
    y_tr = df_tab.loc[train_mask, "label"].values.astype(int)
    X_te = df_tab.loc[test_mask, tab_features].values

    tab_clf = TabPFNClassifier(device=str(device), N_ensemble_configurations=4)
    tab_clf.fit(X_tr, y_tr)
    p_tabpfn_oof[test_mask] = tab_clf.predict_proba(X_te)[:, 1]

tab_auroc = roc_auc_score(y_true, p_tabpfn_oof)
tab_auprc = average_precision_score(y_true, p_tabpfn_oof)
print(f"TabPFN Foundation Model Alone: AUROC = {tab_auroc:.4f} | AUPRC = {tab_auprc:.4f}")

# Sweep Multimodal Blending Alpha
best_alpha = 0.5
best_hybrid_auroc = 0.0

for alpha in [0.0, 0.15, 0.25, 0.35, 0.50, 0.65, 0.85, 1.0]:
    p_blend = alpha * p_dinov2_mean + (1.0 - alpha) * p_tabpfn_oof
    b_auc = roc_auc_score(y_true, p_blend)
    b_pr = average_precision_score(y_true, p_blend)
    print(f"  • Blend Alpha={alpha:.2f} (DINOv2) + {1.0-alpha:.2f} (TabPFN) -> AUROC = {b_auc:.4f} | AUPRC = {b_pr:.4f}")
    if b_auc > best_hybrid_auroc:
        best_hybrid_auroc = b_auc
        best_alpha = alpha

p_hybrid_best = best_alpha * p_dinov2_mean + (1.0 - best_alpha) * p_tabpfn_oof
print("\n" + "=" * 65)
print(f"OPTIMAL MULTIMODAL FOUNDATION ENSEMBLE (Alpha = {best_alpha:.2f}):")
print(f"  • Hybrid AUROC: {roc_auc_score(y_true, p_hybrid_best):.4f}")
print(f"  • Hybrid AUPRC: {average_precision_score(y_true, p_hybrid_best):.4f}")
print("=" * 65)


In [ ]:
# [8/8] Clinical Visualizations & Prediction Export
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, precision_recall_curve

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curve
fpr_d, tpr_d, _ = roc_curve(y_true, p_dinov2_mean)
fpr_h, tpr_h, _ = roc_curve(y_true, p_hybrid_best)

axes[0].plot(fpr_d, tpr_d, color="#2563eb", lw=2.5, label=f"Strategy C (DINOv2 Alone): {np.mean(repeat_aurocs):.3f}")
axes[0].plot(fpr_h, tpr_h, color="#10b981", lw=2.5, label=f"Multimodal Hybrid (DINOv2+TabPFN): {roc_auc_score(y_true, p_hybrid_best):.3f}")
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.4)
axes[0].set_title("ROC Curves across 215 Students (5x5 Locked CV)", fontsize=13, fontweight="bold")
axes[0].set_xlabel("False Positive Rate (1 - Specificity)")
axes[0].set_ylabel("True Positive Rate (Sensitivity)")
axes[0].legend(loc="lower right")
axes[0].grid(True, alpha=0.25)

# Precision-Recall Curve
prec_d, rec_d, _ = precision_recall_curve(y_true, p_dinov2_mean)
prec_h, rec_h, _ = precision_recall_curve(y_true, p_hybrid_best)

axes[1].plot(rec_d, prec_d, color="#2563eb", lw=2.5, label=f"Strategy C (DINOv2 Alone): {np.mean(repeat_auprcs):.3f}")
axes[1].plot(rec_h, prec_h, color="#10b981", lw=2.5, label=f"Multimodal Hybrid: {average_precision_score(y_true, p_hybrid_best):.3f}")
axes[1].axhline(y=40/215, color="k", linestyle="--", alpha=0.4, label=f"Base Rate (18.6%)")
axes[1].set_title("Precision-Recall Curves across 215 Students", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Recall (Sensitivity)")
axes[1].set_ylabel("Precision (PPV)")
axes[1].legend(loc="upper right")
axes[1].grid(True, alpha=0.25)

plt.tight_layout()
plt.savefig("strategy_c_clinical_curves.png", dpi=300)
plt.show()

# Export Out-of-Fold Predictions
df_export = pd.DataFrame({
    "student_id": df_students["student_id"],
    "school": df_students["school"],
    "grade": df_students["grade"],
    "label": y_true,
    "p_dinov2": p_dinov2_mean,
    "p_tabpfn": p_tabpfn_oof,
    "p_hybrid_foundation": p_hybrid_best,
})
df_export.to_csv("p_strategy_c_dinov2_predictions.csv", index=False)
print("Saved predictions to p_strategy_c_dinov2_predictions.csv!")
